In [1]:
import numpy as np
from scipy.signal import butter, filtfilt

def segment_primitives(data, timestamps, fs=100.0):
    """
    最简6轴IMU运动基元分割（纯自我中心坐标系）
    
    输入:
        data: (n,6) ndarray, [ax,ay,az,gx,gy,gz]
              加速度: m/s², 角速度: rad/s (若角速度为deg/s, 将 eps_w 改为 10.0)
        timestamps: (n,) ndarray, 时间戳(秒)
        fs: 采样率(Hz)
    输出:
        list[dict]: 基元列表，每个基元包含自我中心方向/轴、强度、标签
    """
    data = np.asarray(data, dtype=float)
    timestamps = np.asarray(timestamps, dtype=float)
    
    # 参数
    fc = 0.5
    win = int(0.25 * fs)      # 滑动窗口 0.25s
    step = win // 2           # 步长 0.125s
    theta_th = np.deg2rad(35) # 方向突变阈值
    eps_a = 0.1               # 平移噪声阈值 (m/s²)
    eps_w = np.deg2rad(10)    # 旋转噪声阈值 (rad/s)；若角速度为 deg/s 请改为 10.0
    min_dur = 0.15            # 最小基元时长 (s)
    
    n = len(data)
    
    # 1. 重力分离 -> 运动加速度
    b, a = butter(2, fc/(fs/2), 'low')
    gravity = filtfilt(b, a, data[:, :3], axis=0)
    a_body = data[:, :3] - gravity
    
    # 辅助：窗口平均方向（强度不足返回None）
    def mean_dir(signal, idx, w, eps):
        v = signal[idx:idx+w].sum(axis=0)
        nv = np.linalg.norm(v)
        return v / nv if nv > eps * w else None
    
    # 2. 滑动窗口变点检测
    cps = [0]
    for i in range(0, n - win, step):
        j = i + step
        if j + win > n:
            break
        
        # 方向变化
        d1 = mean_dir(a_body, i, win, eps_a)
        d2 = mean_dir(a_body, j, win, eps_a)
        r1 = mean_dir(data[:, 3:], i, win, eps_w)
        r2 = mean_dir(data[:, 3:], j, win, eps_w)
        
        trans_chg = d1 is not None and d2 is not None and \
                    np.arccos(np.clip(d1.dot(d2), -1, 1)) > theta_th
        rot_chg = r1 is not None and r2 is not None and \
                  np.arccos(np.clip(r1.dot(r2), -1, 1)) > theta_th
        
        # 状态突变（静止 <-> 运动）
        ma1 = np.linalg.norm(a_body[i:i+win], axis=1).mean() > eps_a
        ma2 = np.linalg.norm(a_body[j:j+win], axis=1).mean() > eps_a
        mw1 = np.linalg.norm(data[i:i+win, 3:], axis=1).mean() > eps_w
        mw2 = np.linalg.norm(data[j:j+win, 3:], axis=1).mean() > eps_w
        
        if trans_chg or rot_chg or (ma1 != ma2) or (mw1 != mw2):
            cps.append(j)
    
    cps = np.unique(cps)
    if cps[-1] != n - 1:
        cps = np.append(cps, n - 1)
    
    # 3. 提取基元特征
    primitives = []
    for i in range(len(cps) - 1):
        s, e = int(cps[i]), int(cps[i+1])
        dur = timestamps[e] - timestamps[s]
        if dur < min_dur:
            continue
        
        sa = a_body[s:e]
        sw = data[s:e, 3:]
        
        # 平移方向（基元内加速度矢量和的归一化）
        asum = sa.sum(axis=0)
        an = np.linalg.norm(asum)
        t_dir = asum / an if an > eps_a else np.zeros(3)
        t_mag = np.linalg.norm(sa, axis=1).mean()
        
        # 旋转轴（基元内角速度矢量和的归一化）
        wsum = sw.sum(axis=0)
        wn = np.linalg.norm(wsum)
        r_axis = wsum / wn if wn > eps_w else np.zeros(3)
        r_mag = np.linalg.norm(sw, axis=1).mean()
        
        # 标签
        ht, hr = t_mag > eps_a, r_mag > eps_w
        label = 'combined' if (ht and hr) else \
                ('translation' if ht else ('rotation' if hr else 'static'))
        
        primitives.append({
            'start': float(timestamps[s]),
            'end': float(timestamps[e]),
            'duration': float(dur),
            'trans_dir': t_dir.tolist(),   # 自我中心平移方向 (3,)
            'trans_mag': float(t_mag),      # 平移强度
            'rot_axis': r_axis.tolist(),   # 自我中心旋转轴 (3,)
            'rot_mag': float(r_mag),        # 旋转强度
            'label': label
        })
    
    return primitives

In [2]:
# 生成模拟数据（3秒，约100Hz）
n = 300
t = np.linspace(0, 3, n)
data = np.random.randn(n, 6) * 0.05  # 噪声基底
data[:, 2] += 9.8  # 重力在z轴

# 插入一段"沿x轴加速 + 绕z轴旋转"的运动
data[100:150, 0] += 0.5   # x轴加速度
data[100:150, 5] += 2.0  # z轴角速度

# 分割
prims = segment_primitives(data, t, fs=100)
for p in prims:
    print(f"{p['label']:12s} | {p['duration']:.2f}s | "
          f"trans_dir={p['trans_dir']}, rot_axis={p['rot_axis']}")

translation  | 0.36s | trans_dir=[-0.4450584697872604, -0.48047576100169864, -0.7556890905395292], rot_axis=[0.6141617842812143, 0.5631204132782246, -0.552902073497538]
translation  | 0.36s | trans_dir=[-0.9993608897168098, 0.03455262442837571, -0.009161236245069856], rot_axis=[0.9497330764550043, 0.17696352105757637, 0.2582459984244049]
combined     | 0.72s | trans_dir=[0.9988278035616298, 0.027414287841888325, 0.03989330337753633], rot_axis=[0.002749673781129231, -0.00015645737021866677, 0.9999962074004025]
translation  | 0.36s | trans_dir=[-0.9986165569729823, 0.046084369201395144, 0.025321987574752847], rot_axis=[0.6534187204538551, -0.2686102225221703, 0.7077376096527844]
static       | 1.07s | trans_dir=[-0.8355388690319839, -0.5387453817508504, 0.10783418743090535], rot_axis=[-0.5404819512151955, -0.7744933215082342, -0.32869340630709026]
